# 15 — Paper Figures: Preprocessing Evidence

Three panels, each isolating one preprocessing decision: normalization
scheme, borderline-cleaning method, and random-undersampling severity.
Every panel pools **all four classifiers** (8 points: 4 classifiers x 2
runs) rather than picking one "flagship" model — the findings here hold
up the same way, and often more strongly, when averaged across all four.

**Reads:** `./results/*.txt`. **Requires:** notebook 14 run first (style
and loader). **Writes:** `./paper/figures/preprocessing.pdf` / `.png`.


## 1. Load and Preview

In [3]:
# ══════════════════════════════════════════════════════════════
# Confirm the style/loader from notebook 14 are in scope, and preview
# the exact numbers this figure is built from.
# ══════════════════════════════════════════════════════════════

NORM_ORDER  = ["Raw", "Z-score", "Min-max", "Log", "Hybrid"]
NORM_KEYS   = {"Raw": "nonorm", "Z-score": "zscore_all", "Min-max": "minmax_all",
              "Log": "log_all", "Hybrid": "hybrid"}

CLEAN_ORDER = ["None", "ENN", "Tomek", "NearMiss"]
CLEAN_KEYS  = {"None": "hybrid", "ENN": "hybrid_enn", "Tomek": "hybrid_tomek",
              "NearMiss": "hybrid_nearmiss"}

RUS_ORDER   = ["No RUS", "8000", "2000", "500"]
RUS_KEYS    = {"No RUS": "hybrid_tomek", "8000": "tomek_nonsep_8000",
              "2000": "tomek_nonsep_2000", "500": "tomek_nonsep_500"}

print(f"{'Normalization':<14} {'mean TSS':>9}   {'Cleaning':<10} {'mean TSS':>9}   {'RUS':<8} {'mean TSS':>9}")
for i in range(4):
    n_lab = NORM_ORDER[i]; n_v = load_all_classifiers(NORM_KEYS[n_lab])
    c_lab = CLEAN_ORDER[i]; c_v = load_all_classifiers(CLEAN_KEYS[c_lab])
    r_lab = RUS_ORDER[i];  r_v = load_all_classifiers(RUS_KEYS[r_lab])
    print(f"{n_lab:<14} {n_v.mean():>9.3f}   {c_lab:<10} {c_v.mean():>9.3f}   {r_lab:<8} {r_v.mean():>9.3f}")
n_lab = NORM_ORDER[4]; n_v = load_all_classifiers(NORM_KEYS[n_lab])
print(f"{n_lab:<14} {n_v.mean():>9.3f}")


Normalization   mean TSS   Cleaning    mean TSS   RUS       mean TSS
Raw                0.118   None           0.475   No RUS       0.503
Z-score            0.218   ENN            0.475   8000         0.491
Min-max            0.144   Tomek          0.503   2000         0.459
Log                0.451   NearMiss      -0.074   500          0.472
Hybrid             0.475


## Figure — Normalization, Cleaning, and Undersampling

In [4]:
# FIG -- preprocessing evidence: normalization / cleaning / RUS sweep
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.45),
                         gridspec_kw=dict(width_ratios=[1.0, 0.85, 1.15]))

# ---------------------------------------------------------------
# (a) Normalization -- bar + jittered dots, Hybrid is the answer
# ---------------------------------------------------------------
ax = axes[0]
rng = np.random.default_rng(0)
for i, m in enumerate(NORM_ORDER):
    v = load_all_classifiers(NORM_KEYS[m])
    c = TEAL if m == "Hybrid" else "0.62"
    ax.bar(i, v.mean(), 0.60, color=c, edgecolor="none", zorder=3)
    jitter = rng.uniform(-0.09, 0.09, len(v))
    ax.scatter(np.full_like(v, i, dtype=float) + jitter, v, s=5, color="0.15",
              lw=0, alpha=0.75, zorder=5)
    off = 0.055 if v.mean() >= 0 else -0.065
    ax.text(i, v.mean() + off, f"{v.mean():.2f}", ha="center", fontsize=5.9,
            color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.axhline(0, color="0.4", lw=0.6, zorder=4)
ax.set_xticks(range(len(NORM_ORDER)))
ax.set_xticklabels(NORM_ORDER, rotation=25, ha="right", fontsize=6.0)
ax.set_ylim(-0.05, 0.62)
finish(ax, ylab="Test TSS")
ax.set_title("(a)  Normalization", loc="left", fontsize=7.0, pad=4)

# ---------------------------------------------------------------
# (b) Borderline cleaning -- Tomek is the answer, NearMiss is harmful
# ---------------------------------------------------------------
ax = axes[1]
for i, m in enumerate(CLEAN_ORDER):
    v = load_all_classifiers(CLEAN_KEYS[m])
    c = TEAL if m == "Tomek" else (VERM if m == "NearMiss" else "0.62")
    ax.bar(i, v.mean(), 0.60, color=c, edgecolor="none", zorder=3)
    jitter = rng.uniform(-0.09, 0.09, len(v))
    ax.scatter(np.full_like(v, i, dtype=float) + jitter, v, s=5, color="0.15",
              lw=0, alpha=0.75, zorder=5)
    off = 0.055 if v.mean() >= 0 else -0.045
    va = "bottom" if v.mean() >= 0 else "top"
    ax.text(i, v.mean() + off, f"{v.mean():.2f}", ha="center", va=va, fontsize=5.9,
            color=(VERM if m == "NearMiss" else "0.0"), zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.axhline(0, color="0.4", lw=0.6, zorder=4)
ax.set_xticks(range(len(CLEAN_ORDER)))
ax.set_xticklabels(CLEAN_ORDER, rotation=25, ha="right", fontsize=6.0)
ax.set_ylim(-0.24, 0.62)
finish(ax, ylab="Test TSS")
ax.set_title("(b)  Borderline cleaning", loc="left", fontsize=7.0, pad=4)

# ---------------------------------------------------------------
# (c) RUS sweep -- a search over undersampling severity that finds
# nothing: the shaded band is the full 8-point spread at each level,
# the line is the mean. A flat line is the finding.
# ---------------------------------------------------------------
ax = axes[2]
xs = np.arange(len(RUS_ORDER))
means = np.array([load_all_classifiers(RUS_KEYS[r]).mean() for r in RUS_ORDER])
los   = np.array([load_all_classifiers(RUS_KEYS[r]).min() for r in RUS_ORDER])
his   = np.array([load_all_classifiers(RUS_KEYS[r]).max() for r in RUS_ORDER])

ax.fill_between(xs, los, his, color=BLUE, alpha=0.13, lw=0, zorder=2,
                label="4-classifier spread")
ax.plot(xs, means, "-o", color=BLUE, lw=1.1, ms=3.4, mfc="white", mew=0.9,
        zorder=4, label="mean TSS")
best_i = int(np.argmax(means))
ax.scatter([best_i], [means[best_i]], s=40, facecolor="none", edgecolor=TEAL,
          lw=1.2, zorder=5)
ax.annotate("best on average\n(but barely)", xy=(best_i, means[best_i]),
            xytext=(best_i + 0.55, means[best_i] + 0.045), fontsize=5.6,
            color=TEAL, ha="left",
            arrowprops=dict(arrowstyle="-", lw=0.6, color=TEAL))
ax.set_xticks(xs)
ax.set_xticklabels(RUS_ORDER, fontsize=6.0)
ax.set_ylim(0.24, 0.60)
finish(ax, ylab="Test TSS", xlab="Non-SEP target (undersampling amount)")
ax.set_title("(c)  Random undersampling", loc="left", fontsize=7.0, pad=4)
ax.legend(frameon=False, loc="lower left", handlelength=1.0, handletextpad=0.4,
          borderpad=0.1, labelspacing=0.25)

fig.subplots_adjust(left=0.08, right=0.995, top=0.905, bottom=0.265, wspace=0.38)
fig.savefig(f"{FIG_DIR}/preprocessing.pdf")
fig.savefig(f"{FIG_DIR}/preprocessing.png", dpi=340)
plt.show()
print("Saved preprocessing.pdf/png")


Saved preprocessing.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_41597/3240400229.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
